# Praktikum Ekonometrika dan Sains Data — Pertemuan 7–10
**Universitas Padjadjaran | Modul lanjutan regresi**

Notebook terhubung dengan HTML lanjutan. Semua contoh menggunakan **data sintetis**; bukti identifikasi kausal untuk data riil harus dibangun berdasarkan desain dan institusi. Urutan: panel FE/RE → IV/2SLS → DiD → diagnostics → ML → cross-fitting DML.


In [ ]:
%pip -q install linearmodels scikit-learn statsmodels matplotlib seaborn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, RandomEffects
from linearmodels.iv import IV2SLS
from sklearn.model_selection import train_test_split, KFold, cross_validate, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.stats.outliers_influence import variance_inflation_factor
rng = np.random.default_rng(20261014)
print('Library siap.')

## Pertemuan 7 — Panel dan Fixed Effects
Data dibangkitkan dengan efek unit \(c_i\) yang berkorelasi dengan regressor x. Efek kausal benar x adalah 1.5. Pooled OLS dapat bias; FE menyisihkan komponen unit konstan.

In [ ]:
N,T=100,7
unit=np.repeat(np.arange(N),T); year=np.tile(np.arange(T),N)
ability=rng.normal(size=N); common=0.16*year
x=0.9*ability[unit]+0.25*year+rng.normal(size=N*T)
y=1.5*x+2*ability[unit]+common+rng.normal(size=N*T)
panel=pd.DataFrame({'id':unit,'year':year,'x':x,'y':y})
pooled=smf.ols('y ~ x + C(year)', data=panel).fit(cov_type='cluster',cov_kwds={'groups':panel['id']})
panel_ix=panel.set_index(['id','year'])
fe=PanelOLS.from_formula('y ~ 1 + x + EntityEffects + TimeEffects',data=panel_ix).fit(cov_type='clustered',cluster_entity=True)
re=RandomEffects.from_formula('y ~ 1 + x',data=panel_ix).fit(cov_type='clustered',cluster_entity=True)
print('True beta=1.5 | pooled:',round(pooled.params['x'],3),' FE:',round(fe.params['x'],3),' RE:',round(re.params['x'],3))
print(fe.summary)

In [ ]:
# Transformasi within secara manual (satu efek unit; time effects harus ditangani terpisah)
wd=panel.copy();wd['y_w']=wd['y']-wd.groupby('id')['y'].transform('mean');wd['x_w']=wd['x']-wd.groupby('id')['x'].transform('mean')
within=sm.OLS(wd.y_w,wd[['x_w']]).fit(cov_type='cluster',cov_kwds={'groups':wd.id})
print('Within, tanpa time FE:',within.params['x_w'])
print('Perhatikan beda kecil dengan two-way FE akibat efek waktu.')

### Diskusi
1. Mengapa pooled OLS bias pada DGP ini? 2. Mengapa variabel konstan menurut individu tidak dapat diestimasi dalam unit FE? 3. Apa asumsi eksogenitas yang tetap diperlukan oleh FE?

## Pertemuan 8A — Endogeneity, IV, dan 2SLS
X dipengaruhi shock v yang juga berkaitan dengan error u; instrumen z independen dari keduanya. Perhatikan perbedaan first-stage F dan asumsi eksklusi.

In [ ]:
nobs=1200
z=rng.normal(size=nobs);v=rng.normal(size=nobs);e=rng.normal(size=nobs)
u=0.8*v+e
x=0.8*z+v
y=1+2*x+u
ivdf=pd.DataFrame({'y':y,'x':x,'z':z})
ols=smf.ols('y ~ x',ivdf).fit(cov_type='HC3')
iv=IV2SLS.from_formula('y ~ 1 + [x ~ z]',data=ivdf).fit(cov_type='robust')
first=smf.ols('x ~ z',ivdf).fit()
print('True beta=2 | OLS=',round(ols.params['x'],3),'IV=',round(iv.params['x'],3))
print('First-stage F:',first.fvalue,' (heuristik saja)')
print(iv.summary)
print(iv.first_stage)

In [ ]:
# Pelanggaran eksklusi: z memengaruhi y secara langsung
bad=ivdf.copy();bad['y']=ivdf['y']+0.65*ivdf['z']
invalid=IV2SLS.from_formula('y ~ 1 + [x ~ z]',data=bad).fit(cov_type='robust')
print('IV dengan instrumen melanggar exclusion:',round(invalid.params['x'],3))
# Weak IV: penyebut Cov(z,x) kecil -> varians IV membesar
for strength in [1.,0.3,0.05]:
    xx=strength*z+v; yy=1+2*xx+u
    df=pd.DataFrame({'y':yy,'x':xx,'z':z})
    ivx=IV2SLS.from_formula('y ~ 1 + [x ~ z]',data=df).fit(cov_type='robust')
    print('strength',strength,'estimate',round(ivx.params['x'],3),'SE',round(ivx.std_errors['x'],3))

## Pertemuan 8B — Difference-in-Differences dan tren paralel
Panel buatan dua kelompok dan enam periode, pengobatan dimulai pada periode 3. Efek intervensi sebenarnya 4.

In [ ]:
nunits=200;nt=6
uid=np.repeat(np.arange(nunits),nt);tt=np.tile(np.arange(nt),nunits)
treated=(uid<nunits//2).astype(int);post=(tt>=3).astype(int)
a=rng.normal(size=nunits);shock=rng.normal(scale=0.8,size=nunits*nt)
y0=15+2*treated+a[uid]+0.8*tt+shock
y=y0+4*treated*post
did=pd.DataFrame({'id':uid,'t':tt,'treated':treated,'post':post,'y':y})
did['interaction']=did['treated']*did['post']
res=smf.ols('y ~ treated + post + interaction',did).fit(cov_type='cluster',cov_kwds={'groups':did['id']})
print('True ATT=4; two-by-two DiD estimate:',res.params['interaction']);print(res.summary().tables[1])
means=did.groupby(['treated','t']).y.mean().unstack(0)
means.rename(columns={0:'Control',1:'Treated'}).plot(marker='o',title='Treated dan kontrol sepanjang waktu');plt.axvline(2.5,linestyle='--');plt.xlabel('Periode');plt.ylabel('Rata-rata Y');plt.show()

In [ ]:
# Event-study: koefisien interaksi tiap periode relatif ke t=2
for k in [0,1,3,4,5]:
    did[f'event_{k}']=((did['t']==k)&(did['treated']==1)).astype(int)
terms=' + '.join(f'event_{k}' for k in [0,1,3,4,5])
event=smf.ols('y ~ C(id) + C(t) + '+terms,did).fit(cov_type='cluster',cov_kwds={'groups':did['id']})
print(event.params[[f'event_{k}' for k in [0,1,3,4,5]]].round(3))
# Placebo: pre-treatment sample t < 3, tempatkan pseudo post di t >= 1
pre=did.query('t < 3').copy();pre['placebo']=(pre['treated']*(pre['t']>=1)).astype(int)
placebo=smf.ols('y ~ C(id) + C(t) + placebo',pre).fit(cov_type='cluster',cov_kwds={'groups':pre['id']})
print('Placebo pre-period:',placebo.params['placebo'],'p=',placebo.pvalues['placebo'])

## Pertemuan 9 — Diagnostic tests dan robustness
Pada cross-section heteroskedastik: periksa Breusch–Pagan, VIF, HC3. Pada panel dan DiD gunakan clustering sesuai desain, bukan HC3 saja.

In [ ]:
ns=650
x1=rng.normal(size=ns);x2=0.8*x1+rng.normal(scale=0.5,size=ns)
eps=rng.normal(size=ns)*(0.4+0.9*np.abs(x1))
y=3+1.2*x1+0.9*x2+eps
df=pd.DataFrame({'y':y,'x1':x1,'x2':x2})
base=smf.ols('y ~ x1 + x2',df).fit();robust=base.get_robustcov_results(cov_type='HC3')
print('Classic SE:',base.bse.to_dict());print('HC3 SE:',dict(zip(base.params.index,robust.bse)))
print('Breusch–Pagan (LM,p,F,p):',het_breuschpagan(base.resid,base.model.exog))
X=sm.add_constant(df[['x1','x2']]);print('VIF:',{X.columns[i]:variance_inflation_factor(X.values,i) for i in [1,2]})
fig,ax=plt.subplots(figsize=(7,3));ax.scatter(base.fittedvalues,base.resid,alpha=.4);ax.axhline(0,color='black');ax.set(xlabel='Fitted',ylabel='Residual');plt.show()

In [ ]:
# Sensitivitas terhadap korelasi dalam unit: standar error konvensional vs cluster
plain=smf.ols('y ~ treated + post + interaction',did).fit()
cluster=smf.ols('y ~ treated + post + interaction',did).fit(cov_type='cluster',cov_kwds={'groups':did.id})
print(pd.DataFrame({'nonclustered':plain.bse,'clustered_unit':cluster.bse}))

## Pertemuan 10 — Machine Learning untuk prediksi
Agar tidak terjadi kebocoran data, scaling dilakukan di dalam pipeline. Hyperparameter ditentukan pada training, lalu dievaluasi pada test. Model pada data nonlinear berikut hanya ditujukan untuk prediksi.

In [ ]:
from sklearn.model_selection import GridSearchCV
nobs=900
x1=rng.uniform(-3,3,nobs);x2=rng.normal(size=nobs);x3=rng.normal(size=nobs)
y=2*np.sin(x1)+0.8*x2**2-0.5*x3+rng.normal(scale=0.5,size=nobs)
X=pd.DataFrame({'x1':x1,'x2':x2,'x3':x3})
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=12)
models={
 'OLS':LinearRegression(),
 'Ridge':make_pipeline(StandardScaler(),RidgeCV(alphas=np.logspace(-3,3,40))),
 'Lasso':make_pipeline(StandardScaler(),LassoCV(cv=5,random_state=12,max_iter=5000)),
 'RandomForest':RandomForestRegressor(n_estimators=150,min_samples_leaf=3,random_state=12,n_jobs=-1)
}
rows=[]
for name,m in models.items():
 m.fit(Xtr,ytr);pred=m.predict(Xte)
 rows.append({'Model':name,'RMSE':np.sqrt(mean_squared_error(yte,pred)),'MAE':mean_absolute_error(yte,pred)})
print(pd.DataFrame(rows).sort_values('RMSE').to_string(index=False))

### Group-aware CV
Gunakan GroupKFold ketika deployment mengharuskan generalisasi pada grup yang belum pernah terlihat. Untuk time-series gunakan TimeSeriesSplit sesuai struktur horizon prediksi.

In [ ]:
groups=np.repeat(np.arange(90),10)
cv=GroupKFold(n_splits=5)
model=RandomForestRegressor(n_estimators=70,min_samples_leaf=4,random_state=0,n_jobs=-1)
sc=cross_validate(model,X,y,groups=groups,cv=cv,scoring='neg_root_mean_squared_error')
print('GroupKFold RMSE rata-rata:',-sc['test_score'].mean())

## Cross-fitting DML: partially linear, perlakuan kontinu
Dengan \(D=m_0(X)+v\), \(Y=	heta D+g_0(X)+\epsilon\), serta eksogenitas yang sesuai, residualisasikan Y dan D pada fold berbeda. Implementasi berikut adalah ilustrasi titik estimasi, **bukan** standard error DML formal. Untuk inference gunakan paket khusus dan syarat regularitas.

In [ ]:
nx=900
W=rng.normal(size=(nx,3));v=rng.normal(size=nx);eps=rng.normal(size=nx)
D=0.7*W[:,0]**2+0.6*np.sin(W[:,1])+v
Y=1.8*D+1.2*W[:,0]-0.5*W[:,1]**2+0.7*np.sin(W[:,2])+eps
Dres=np.zeros(nx);Yres=np.zeros(nx)
cv=KFold(n_splits=3,shuffle=True,random_state=10)
for train,test in cv.split(W):
  fitD=RandomForestRegressor(n_estimators=90,min_samples_leaf=5,random_state=12,n_jobs=-1).fit(W[train],D[train])
  fitY=RandomForestRegressor(n_estimators=90,min_samples_leaf=5,random_state=13,n_jobs=-1).fit(W[train],Y[train])
  Dres[test]=D[test]-fitD.predict(W[test]);Yres[test]=Y[test]-fitY.predict(W[test])
theta=np.dot(Dres,Yres)/np.dot(Dres,Dres)
print('True theta=1.8 | Cross-fitted point estimate:',round(theta,3))
print('Keterbatasan: bukan CI valid tanpa estimator varians dan asumsi DML formal.')

## Kerangka laporan penugasan
**Model v1 (10%):** tujuan/estimand, diagram asumsi, spesifikasi, output estimator utama beserta CI dan risiko identifikasi.

**Milestone B (10%):** rencana pembagian data sesuai struktur, pemilihan metrik, weak IV/parallel trends/placebo bila relevan, robust SE, sensitivitas, anti-leakage, dan reproduce-able code.

**Tugas refleksi:** model mana yang digunakan untuk menjawab asosiasi, prediksi, ATE/ATT/LATE? Jelaskan alasan pemilihan, bukan semata p-value atau RMSE.
